# Notebook 6: Introduction to Detection and Segmentation Concepts

---

## Learning Objectives

By the end of this notebook, you will:

- Understand the key **object detection** concepts: bounding boxes, IoU, anchor boxes, and mAP
- Know the high-level architecture of **YOLO**, **SSD**, and **Faster R-CNN**
- Understand **semantic segmentation** and architectures like **FCN** and **U-Net**
- Know what **instance segmentation** is and how **Mask R-CNN** approaches it
- Be able to **calculate IoU** and **visualize bounding boxes** programmatically

## Prerequisites

- **DL300 Notebooks 01-05**: CNN fundamentals, transfer learning, fine-tuning
- **PyTorch basics**: Tensors, basic operations
- **Matplotlib**: Basic plotting

> **Note**: This is a **conceptual** notebook. We will not train any detection or segmentation models here. The goal is to build intuition and vocabulary before diving into these topics in a dedicated course or project.

## Table of Contents

1. [Setup & Imports](#1.-Setup-&-Imports)
2. [From Classification to Detection](#2.-From-Classification-to-Detection)
3. [Bounding Boxes & Representations](#3.-Bounding-Boxes-&-Representations)
4. [Intersection over Union (IoU)](#4.-Intersection-over-Union-(IoU))
5. [Anchor Boxes](#5.-Anchor-Boxes)
6. [Object Detection Architectures Overview](#6.-Object-Detection-Architectures-Overview)
7. [Mean Average Precision (mAP)](#7.-Mean-Average-Precision-(mAP))
8. [Semantic Segmentation Overview](#8.-Semantic-Segmentation-Overview)
9. [Instance Segmentation (Mask R-CNN)](#9.-Instance-Segmentation-(Mask-R-CNN))
10. [Code: Visualize Bounding Boxes](#10.-Code:-Visualize-Bounding-Boxes)
11. [Code: IoU Calculation](#11.-Code:-IoU-Calculation)
12. [Common Mistakes & Debugging Tips](#12.-Common-Mistakes-&-Debugging-Tips)
13. [Exercises](#13.-Exercises)
14. [Exercise Solutions](#14.-Exercise-Solutions)

---

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch

print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

---

## 2. From Classification to Detection

So far, we have worked with **image classification**: one image in, one label out.

Computer vision tasks form a hierarchy of increasing complexity:

| Task | Input | Output | Example |
|------|-------|--------|---------|
| **Classification** | Image | Single class label | "cat" |
| **Localization** | Image | Class label + bounding box | "cat" at (x, y, w, h) |
| **Object Detection** | Image | Multiple labels + bounding boxes | "cat" at box1, "dog" at box2 |
| **Semantic Segmentation** | Image | Per-pixel class labels | Every pixel labeled |
| **Instance Segmentation** | Image | Per-pixel labels + instance IDs | Each cat gets its own mask |

```
Classification    Localization     Detection      Semantic Seg    Instance Seg
+----------+    +----------+    +----------+    +----------+    +----------+
|          |    |  +----+  |    | +--+ +--+|    |##########|    |##  @@@@@@|
|   CAT    |    |  |CAT |  |    | |CA| |DO||    |##CAT#####|    |##CA@@DOG@|
|          |    |  +----+  |    | +--+ +--+|    |##########|    |##TT@@@@@@|
+----------+    +----------+    +----------+    +----------+    +----------+
 "cat"           "cat" + box    boxes + labels   pixel labels    masks + IDs
```

---

## 3. Bounding Boxes & Representations

A **bounding box** is a rectangle that tightly encloses an object.

### Common Formats

| Format | Description | Values |
|--------|-------------|--------|
| **(x1, y1, x2, y2)** | Top-left and bottom-right corners | Also called `xyxy` |
| **(x, y, w, h)** | Top-left corner + width & height | Also called `xywh` |
| **(cx, cy, w, h)** | Center + width & height | Used by YOLO, SSD |

### Coordinate Conventions

- Origin is **top-left** corner of the image
- $x$ increases to the right, $y$ increases downward
- Coordinates can be **absolute** (pixels) or **normalized** (0 to 1)

### Conversion Formulas

From **(cx, cy, w, h)** to **(x1, y1, x2, y2)**:

$$x_1 = c_x - \frac{w}{2}, \quad y_1 = c_y - \frac{h}{2}$$
$$x_2 = c_x + \frac{w}{2}, \quad y_2 = c_y + \frac{h}{2}$$

From **(x1, y1, x2, y2)** to **(cx, cy, w, h)**:

$$c_x = \frac{x_1 + x_2}{2}, \quad c_y = \frac{y_1 + y_2}{2}$$
$$w = x_2 - x_1, \quad h = y_2 - y_1$$

In [ ]:
# Bounding box format conversions

def xyxy_to_cxcywh(box):
    """Convert (x1, y1, x2, y2) to (cx, cy, w, h)."""
    x1, y1, x2, y2 = box
    return [
        (x1 + x2) / 2,  # cx
        (y1 + y2) / 2,  # cy
        x2 - x1,        # w
        y2 - y1,        # h
    ]


def cxcywh_to_xyxy(box):
    """Convert (cx, cy, w, h) to (x1, y1, x2, y2)."""
    cx, cy, w, h = box
    return [
        cx - w / 2,  # x1
        cy - h / 2,  # y1
        cx + w / 2,  # x2
        cy + h / 2,  # y2
    ]


# Example
box_xyxy = [50, 30, 200, 180]
box_cxcywh = xyxy_to_cxcywh(box_xyxy)
box_back = cxcywh_to_xyxy(box_cxcywh)

print(f"Original (xyxy):   {box_xyxy}")
print(f"Converted (cxcywh): {box_cxcywh}")
print(f"Back to (xyxy):    {box_back}")
assert box_xyxy == box_back, "Round-trip conversion failed!"
print("Round-trip conversion successful.")

---

## 4. Intersection over Union (IoU)

**IoU** (also called the **Jaccard index**) measures how much two bounding boxes overlap. It is the standard metric for evaluating object detection.

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

### Properties

- $\text{IoU} \in [0, 1]$
- $\text{IoU} = 0$: No overlap
- $\text{IoU} = 1$: Perfect overlap
- Typical threshold for a "correct" detection: $\text{IoU} \geq 0.5$

### Visual Intuition

```
  +--------+
  |   A    |
  |    +---+----+
  |    |///|    |       /// = Intersection
  +----+---+    |       A + B - /// = Union
       |   B    |
       +--------+

  IoU = Area(///) / Area(A + B - ///)
```

### Computing IoU Step by Step

Given boxes $A = (x_1^A, y_1^A, x_2^A, y_2^A)$ and $B = (x_1^B, y_1^B, x_2^B, y_2^B)$:

1. **Intersection rectangle**:
   - $x_1^I = \max(x_1^A, x_1^B)$, $y_1^I = \max(y_1^A, y_1^B)$
   - $x_2^I = \min(x_2^A, x_2^B)$, $y_2^I = \min(y_2^A, y_2^B)$

2. **Intersection area**:
   - $w^I = \max(0, x_2^I - x_1^I)$, $h^I = \max(0, y_2^I - y_1^I)$
   - $\text{Area}_I = w^I \times h^I$

3. **Union area**:
   - $\text{Area}_A = (x_2^A - x_1^A)(y_2^A - y_1^A)$
   - $\text{Area}_B = (x_2^B - x_1^B)(y_2^B - y_1^B)$
   - $\text{Area}_U = \text{Area}_A + \text{Area}_B - \text{Area}_I$

4. **IoU** = $\text{Area}_I / \text{Area}_U$

---

## 5. Anchor Boxes

**Anchor boxes** (also called **prior boxes** or **default boxes**) are predefined bounding boxes of various sizes and aspect ratios, placed at each spatial location in the feature map.

### Why Anchor Boxes?

- Detecting objects of arbitrary size and shape is hard
- Instead of predicting boxes from scratch, the network predicts **offsets** relative to anchor boxes
- This makes the regression problem easier and training more stable

### How They Work

1. Place a grid of anchor boxes at each position in the feature map
2. Each anchor has a predefined size and aspect ratio (e.g., 1:1, 1:2, 2:1)
3. The network predicts, for each anchor:
   - **Classification**: Is there an object? If so, which class?
   - **Regression**: How to adjust (shift + scale) the anchor to fit the object?

### Example Anchors

```
Feature map (e.g., 7x7):
Each cell gets K anchor boxes (e.g., K=9 for 3 scales x 3 aspect ratios)

+---+---+---+---+---+---+---+
| K | K | K | K | K | K | K |    Total anchors = 7 x 7 x 9 = 441
+---+---+---+---+---+---+---+
| K | K | K | K | K | K | K |    Most will be classified as "background"
+---+---+---+---+---+---+---+    Only a few match actual objects
| ...                       |
```

### Matching Anchors to Ground Truth

- An anchor is a **positive match** if $\text{IoU}(\text{anchor}, \text{ground truth}) > 0.5$
- An anchor is a **negative match** (background) if $\text{IoU} < 0.3$
- Anchors in between are typically ignored during training

---

## 6. Object Detection Architectures Overview

### Two-Stage Detectors

#### Faster R-CNN (2015)

A two-stage detector that first proposes regions, then classifies them.

```
Image -> Backbone CNN -> Feature Map
                              |
                    +---------+---------+
                    |                   |
            Stage 1: RPN          Stage 2: Classification
        (Region Proposal Net)     (ROI Pooling + FC layers)
                    |                   |
            ~2000 proposals       Final boxes + classes
```

- **Stage 1 (RPN)**: Scans the feature map with anchors and proposes regions likely to contain objects
- **Stage 2**: Crops features for each proposal (ROI Pooling), then classifies and refines the box
- **Pros**: High accuracy
- **Cons**: Slower than single-stage methods

### One-Stage (Single-Shot) Detectors

#### YOLO (You Only Look Once, 2016+)

Divides the image into a grid and predicts boxes + classes in a single forward pass.

```
Image -> Backbone CNN -> Feature Map (S x S grid)
                              |
                    Each cell predicts:
                    - B bounding boxes (cx, cy, w, h, confidence)
                    - C class probabilities
                              |
                    Non-Maximum Suppression (NMS)
                              |
                    Final detections
```

- **Key idea**: Detection as a single regression problem
- **Pros**: Very fast (real-time capable)
- **Cons**: Can struggle with small objects and objects close together
- **Versions**: YOLOv1 through YOLOv8+ (each version improved on the last)

#### SSD (Single Shot MultiBox Detector, 2016)

Uses **multi-scale feature maps** to detect objects at different sizes.

```
Image -> Backbone CNN -> Multiple feature maps at different scales
                              |
                    +----+----+----+----+
                    |    |    |    |    |
                  38x38 19x19 10x10 5x5  ...  (decreasing resolution)
                  small  |   medium  |    large objects
                objects  |           |    
                    +----+----+----+----+
                              |
                    NMS -> Final detections
```

- **Key idea**: Detect at multiple scales from different layers of the backbone
- **Pros**: Good balance of speed and accuracy
- **Cons**: Lower accuracy than Faster R-CNN on small objects

### Summary Comparison

| Method | Type | Speed | Accuracy | Key Idea |
|--------|------|-------|----------|----------|
| Faster R-CNN | Two-stage | Moderate | High | Region proposals + refinement |
| YOLO | Single-stage | Very fast | Good | Grid-based single pass |
| SSD | Single-stage | Fast | Good | Multi-scale feature maps |

---

## 7. Mean Average Precision (mAP)

**mAP** is the standard evaluation metric for object detection.

### Step-by-Step Computation

1. **For each detection**, compute IoU with all ground-truth boxes
2. **Match detections** to ground-truth boxes (IoU > threshold, typically 0.5)
3. **Classify each detection** as True Positive (TP) or False Positive (FP)
4. **Compute Precision and Recall** at different confidence thresholds:

$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}, \quad \text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$

5. **Plot the Precision-Recall curve** for each class
6. **AP (Average Precision)** = Area under the PR curve for one class
7. **mAP** = Mean of AP across all classes:

$$\text{mAP} = \frac{1}{C} \sum_{c=1}^{C} \text{AP}_c$$

### Common Variants

- **mAP@0.5**: IoU threshold = 0.5 (PASCAL VOC standard)
- **mAP@0.75**: Stricter IoU threshold = 0.75
- **mAP@[0.5:0.95]**: Average over IoU thresholds from 0.5 to 0.95 in steps of 0.05 (COCO standard)

---

## 8. Semantic Segmentation Overview

**Semantic segmentation** assigns a class label to every pixel in the image.

### Pixel-Wise Classification

- Input: Image of shape $(H, W, 3)$
- Output: Label map of shape $(H, W)$ where each value is a class index
- Or equivalently, a probability map of shape $(C, H, W)$ where $C$ is the number of classes

### Key Architectures

#### Fully Convolutional Network (FCN, 2015)

The foundational architecture for semantic segmentation.

```
Image (H x W x 3)
      |
  [Encoder: standard CNN backbone]
  Spatial resolution decreases, channels increase
      |
  [1x1 convolutions for per-pixel classification]
      |
  [Upsampling / transposed convolutions]
  Spatial resolution increases back to H x W
      |
  [Skip connections from encoder]
      |
  Output (H x W x C)
```

- **Key insight**: Replace FC layers with 1x1 convolutions to preserve spatial information
- **Skip connections**: Combine coarse high-level features with fine low-level features
- **Loss**: Cross-entropy loss per pixel

#### U-Net (2015)

Originally designed for biomedical image segmentation, now widely used.

```
Encoder (Contracting Path)        Decoder (Expanding Path)

[Input]  ---(skip connection)---  [Output]
   |                                  ^
 [Conv+Pool]  ---(skip)---      [UpConv+Conv]
   |                                  ^
 [Conv+Pool]  ---(skip)---      [UpConv+Conv]
   |                                  ^
 [Conv+Pool]  ---(skip)---      [UpConv+Conv]
   |                                  ^
   +-------->  [Bottleneck]  ---------+
```

- **Symmetric encoder-decoder** with skip connections at every level
- Skip connections **concatenate** (not add) encoder features with decoder features
- Excellent for tasks where precise localization is critical
- Works well even with limited training data

### Segmentation Loss

- **Pixel-wise cross-entropy**: Standard approach

$$\mathcal{L} = -\frac{1}{H \times W} \sum_{i=1}^{H} \sum_{j=1}^{W} \sum_{c=1}^{C} y_{i,j,c} \log(\hat{y}_{i,j,c})$$

- **Dice loss**: Better for imbalanced classes (e.g., small lesions)

$$\text{Dice}(A, B) = \frac{2|A \cap B|}{|A| + |B|}$$

---

## 9. Instance Segmentation (Mask R-CNN)

**Instance segmentation** combines object detection with semantic segmentation: it detects each object AND produces a pixel-level mask for each instance.

### Mask R-CNN (2017)

Extends Faster R-CNN by adding a **mask prediction branch**.

```
Image -> Backbone CNN + FPN -> Feature Maps
                                    |
                          Region Proposal Network
                                    |
                            ROI Align (not ROI Pool)
                                    |
                    +---------------+---------------+
                    |               |               |
              Classification    Box Regression    Mask Branch
              (class label)    (box offsets)     (binary mask per class)
```

- **ROI Align** (instead of ROI Pool): Avoids quantization, preserving spatial precision
- **Mask branch**: Small FCN that predicts a binary mask for each detected object
- Predicts masks **per class**, then selects the mask corresponding to the predicted class

### Comparison of Segmentation Tasks

| Task | Distinguishes instances? | Pixel-level? | Example output |
|------|------------------------|--------------|----------------|
| Semantic Segmentation | No | Yes | All cats = same color |
| Instance Segmentation | Yes | Yes | Cat 1 = blue, Cat 2 = red |
| Panoptic Segmentation | Yes (things) + No (stuff) | Yes | Cats + sky + road all labeled |

---

## 10. Code: Visualize Bounding Boxes

Let us create a synthetic image and draw bounding boxes on it.

In [ ]:
np.random.seed(42)

def create_synthetic_scene(img_h=300, img_w=400):
    """Create a synthetic image with colored rectangles as 'objects'.
    
    Returns:
        image: numpy array (H, W, 3), dtype float in [0, 1]
        objects: list of dicts with 'label', 'bbox' (x1, y1, x2, y2), 'color'
    """
    # Background: light gray with some noise
    image = np.ones((img_h, img_w, 3)) * 0.85
    image += np.random.randn(img_h, img_w, 3) * 0.03
    image = np.clip(image, 0, 1)
    
    objects = [
        {'label': 'cat',  'bbox': [30, 50, 170, 200],  'color': [0.8, 0.3, 0.2]},
        {'label': 'dog',  'bbox': [200, 80, 350, 250], 'color': [0.2, 0.5, 0.8]},
        {'label': 'bird', 'bbox': [280, 20, 370, 90],  'color': [0.3, 0.8, 0.3]},
    ]
    
    # Draw filled rectangles as synthetic objects
    for obj in objects:
        x1, y1, x2, y2 = obj['bbox']
        image[y1:y2, x1:x2] = obj['color']
    
    return image, objects


image, objects = create_synthetic_scene()
print(f"Image shape: {image.shape}")
print(f"Objects: {[o['label'] for o in objects]}")

In [ ]:
def draw_bounding_boxes(image, detections, ground_truth=None, title="Bounding Boxes"):
    """Draw bounding boxes on an image.
    
    Args:
        image: numpy array (H, W, 3)
        detections: list of dicts with 'label', 'bbox' [x1, y1, x2, y2], 
                    optionally 'confidence'
        ground_truth: optional list of dicts with 'label', 'bbox' (drawn as dashed)
        title: plot title
    """
    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(image)
    
    # Color map for labels
    unique_labels = list(set(d['label'] for d in detections))
    cmap = plt.cm.Set1
    label_colors = {lbl: cmap(i / max(len(unique_labels), 1)) for i, lbl in enumerate(unique_labels)}
    
    # Draw ground truth (dashed green boxes)
    if ground_truth is not None:
        for gt in ground_truth:
            x1, y1, x2, y2 = gt['bbox']
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor='green', facecolor='none', linestyle='--'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, f"GT: {gt['label']}", color='green',
                    fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
    
    # Draw detections (solid colored boxes)
    for det in detections:
        x1, y1, x2, y2 = det['bbox']
        color = label_colors[det['label']]
        
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=3, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Label text
        conf_str = f" ({det['confidence']:.2f})" if 'confidence' in det else ""
        ax.text(x1, y1 - 5, f"{det['label']}{conf_str}", color=color,
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
    
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Ground-truth boxes (the actual objects)
ground_truth = objects

# Simulated "predicted" detections (slightly offset, with confidence scores)
detections = [
    {'label': 'cat',  'bbox': [25, 45, 175, 205], 'confidence': 0.92},
    {'label': 'dog',  'bbox': [195, 75, 355, 255], 'confidence': 0.87},
    {'label': 'bird', 'bbox': [275, 15, 375, 95],  'confidence': 0.78},
    {'label': 'cat',  'bbox': [100, 150, 200, 270], 'confidence': 0.35},  # False positive
]

draw_bounding_boxes(image, detections, ground_truth=ground_truth,
                    title="Object Detection: Predictions vs Ground Truth")

---

## 11. Code: IoU Calculation

In [ ]:
def compute_iou(box_a, box_b):
    """Compute Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box_a: list or array [x1, y1, x2, y2]
        box_b: list or array [x1, y1, x2, y2]
    
    Returns:
        iou: float in [0, 1]
    """
    # Compute intersection rectangle
    x1_i = max(box_a[0], box_b[0])
    y1_i = max(box_a[1], box_b[1])
    x2_i = min(box_a[2], box_b[2])
    y2_i = min(box_a[3], box_b[3])
    
    # Intersection area (0 if no overlap)
    inter_width = max(0, x2_i - x1_i)
    inter_height = max(0, y2_i - y1_i)
    inter_area = inter_width * inter_height
    
    # Areas of each box
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    
    # Union area
    union_area = area_a + area_b - inter_area
    
    # IoU
    if union_area == 0:
        return 0.0
    
    iou = inter_area / union_area
    return iou


# Test with known cases
print("=== IoU Test Cases ===")

# Case 1: Perfect overlap
iou1 = compute_iou([0, 0, 100, 100], [0, 0, 100, 100])
print(f"Perfect overlap:    IoU = {iou1:.4f}  (expected: 1.0000)")

# Case 2: No overlap
iou2 = compute_iou([0, 0, 50, 50], [60, 60, 100, 100])
print(f"No overlap:         IoU = {iou2:.4f}  (expected: 0.0000)")

# Case 3: Partial overlap
iou3 = compute_iou([0, 0, 100, 100], [50, 50, 150, 150])
print(f"Partial overlap:    IoU = {iou3:.4f}  (expected: ~0.1429)")

# Case 4: One box inside another
iou4 = compute_iou([0, 0, 100, 100], [25, 25, 75, 75])
print(f"Box inside another: IoU = {iou4:.4f}  (expected: 0.2500)")

In [ ]:
def visualize_iou(box_a, box_b, title=None):
    """Visualize two bounding boxes and their IoU."""
    iou = compute_iou(box_a, box_b)
    
    fig, ax = plt.subplots(1, figsize=(6, 6))
    
    # Determine plot limits
    all_coords = box_a + box_b
    margin = 20
    ax.set_xlim(min(all_coords) - margin, max(all_coords) + margin)
    ax.set_ylim(max(all_coords) + margin, min(all_coords) - margin)  # Invert y-axis
    
    # Draw box A
    rect_a = patches.Rectangle(
        (box_a[0], box_a[1]), box_a[2] - box_a[0], box_a[3] - box_a[1],
        linewidth=2, edgecolor='blue', facecolor='blue', alpha=0.3, label='Box A'
    )
    ax.add_patch(rect_a)
    
    # Draw box B
    rect_b = patches.Rectangle(
        (box_b[0], box_b[1]), box_b[2] - box_b[0], box_b[3] - box_b[1],
        linewidth=2, edgecolor='red', facecolor='red', alpha=0.3, label='Box B'
    )
    ax.add_patch(rect_b)
    
    # Highlight intersection
    x1_i = max(box_a[0], box_b[0])
    y1_i = max(box_a[1], box_b[1])
    x2_i = min(box_a[2], box_b[2])
    y2_i = min(box_a[3], box_b[3])
    
    if x2_i > x1_i and y2_i > y1_i:
        rect_i = patches.Rectangle(
            (x1_i, y1_i), x2_i - x1_i, y2_i - y1_i,
            linewidth=2, edgecolor='green', facecolor='green', alpha=0.5,
            label=f'Intersection'
        )
        ax.add_patch(rect_i)
    
    title_str = title or f"IoU = {iou:.4f}"
    ax.set_title(title_str, fontsize=14)
    ax.legend(fontsize=11)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return iou


# Visualize different IoU scenarios
print("--- High overlap ---")
visualize_iou([10, 10, 110, 110], [30, 30, 130, 130], title="High Overlap")

print("--- Low overlap ---")
visualize_iou([10, 10, 80, 80], [60, 60, 150, 150], title="Low Overlap")

print("--- No overlap ---")
visualize_iou([10, 10, 50, 50], [80, 80, 140, 140], title="No Overlap")

In [ ]:
# Compute IoU between our predicted detections and ground truth

print("IoU between predictions and ground truth:")
print("-" * 50)

for det in detections:
    best_iou = 0.0
    best_gt = None
    for gt in ground_truth:
        iou = compute_iou(det['bbox'], gt['bbox'])
        if iou > best_iou:
            best_iou = iou
            best_gt = gt['label']
    
    match_str = "MATCH" if best_iou >= 0.5 else "NO MATCH"
    print(f"  Pred: {det['label']:5s} (conf={det['confidence']:.2f}) | "
          f"Best GT: {best_gt or 'none':5s} | IoU={best_iou:.4f} | {match_str}")

---

## 12. Common Mistakes & Debugging Tips

### Mistake 1: Mixing up bounding box formats

Different frameworks use different conventions:
- **COCO**: `[x, y, width, height]` (top-left + size)
- **Pascal VOC**: `[x1, y1, x2, y2]` (corners)
- **YOLO**: `[cx, cy, w, h]` normalized to [0, 1]

**Fix**: Always check the format expected by your framework and convert explicitly.

### Mistake 2: Forgetting that y-axis is inverted in images

In images, (0, 0) is the **top-left** corner. $y$ increases downward. This is the opposite of standard math coordinates.

### Mistake 3: IoU calculation with zero-area boxes

If a box has zero width or height, the union area is zero, causing a division-by-zero error.

**Fix**: Add a guard clause (as we did in our implementation).

### Mistake 4: Confusing semantic and instance segmentation

- Semantic: All cats get the same class label. Cannot distinguish individual cats.
- Instance: Each cat gets its own unique mask.

### Mistake 5: Not applying Non-Maximum Suppression (NMS)

Detectors often produce many overlapping boxes for the same object. NMS removes duplicates by:
1. Sort detections by confidence
2. Keep the highest-confidence detection
3. Remove all other detections with IoU > threshold (e.g., 0.5) with the kept detection
4. Repeat for the next highest-confidence detection

### Debugging Checklist

1. **Visualize your bounding boxes** before and after any coordinate transformation
2. **Print box coordinates** and verify they make sense (no negative widths, boxes within image bounds)
3. **Check IoU values** between predictions and ground truth -- they should be > 0.5 for good detections
4. **Verify label encoding** matches between your dataset and model output

---

## 13. Exercises

### Exercise 1: Calculate IoU for Given Bounding Boxes

Compute the IoU for the following pairs of bounding boxes (format: `[x1, y1, x2, y2]`):

- **Pair A**: `[10, 20, 80, 90]` and `[50, 60, 120, 130]`
- **Pair B**: `[0, 0, 200, 200]` and `[50, 50, 100, 100]`
- **Pair C**: `[10, 10, 50, 50]` and `[50, 50, 90, 90]` (corner-touching)

### Exercise 2: Vectorized IoU

Implement a vectorized `compute_iou_matrix` function that takes:
- `boxes_a`: numpy array of shape `(N, 4)` 
- `boxes_b`: numpy array of shape `(M, 4)`

And returns an `(N, M)` IoU matrix where entry `[i, j]` is the IoU between `boxes_a[i]` and `boxes_b[j]`.

### Exercise 3: Simple NMS Implementation

Implement Non-Maximum Suppression (NMS):
- Input: list of detections (each with `bbox` and `confidence`), IoU threshold
- Output: filtered list of detections after suppression

---

## 14. Exercise Solutions

### Solution 1: Calculate IoU for Given Bounding Boxes

In [ ]:
# Pair A
iou_a = compute_iou([10, 20, 80, 90], [50, 60, 120, 130])
print(f"Pair A: IoU = {iou_a:.4f}")

# Manual verification for Pair A:
# Intersection: x1=50, y1=60, x2=80, y2=90 -> w=30, h=30 -> area=900
# Area A: 70 * 70 = 4900
# Area B: 70 * 70 = 4900
# Union: 4900 + 4900 - 900 = 8900
# IoU: 900 / 8900 = 0.1011
print(f"  Manual check: 900 / 8900 = {900/8900:.4f}")

# Pair B
iou_b = compute_iou([0, 0, 200, 200], [50, 50, 100, 100])
print(f"\nPair B: IoU = {iou_b:.4f}")
# Intersection = smaller box = 50*50 = 2500
# Area A = 200*200 = 40000
# Area B = 50*50 = 2500
# Union = 40000 + 2500 - 2500 = 40000
# IoU = 2500/40000 = 0.0625
print(f"  Manual check: 2500 / 40000 = {2500/40000:.4f}")

# Pair C
iou_c = compute_iou([10, 10, 50, 50], [50, 50, 90, 90])
print(f"\nPair C: IoU = {iou_c:.4f}")
# Intersection: x1=50, y1=50, x2=50, y2=50 -> w=0, h=0 -> area=0
# IoU = 0
print(f"  Corner-touching boxes have zero intersection.")

### Solution 2: Vectorized IoU

In [ ]:
def compute_iou_matrix(boxes_a, boxes_b):
    """Compute pairwise IoU between two sets of boxes (vectorized).
    
    Args:
        boxes_a: numpy array of shape (N, 4), format [x1, y1, x2, y2]
        boxes_b: numpy array of shape (M, 4), format [x1, y1, x2, y2]
    
    Returns:
        iou_matrix: numpy array of shape (N, M)
    """
    boxes_a = np.array(boxes_a, dtype=float)
    boxes_b = np.array(boxes_b, dtype=float)
    
    N = boxes_a.shape[0]
    M = boxes_b.shape[0]
    
    # Expand dimensions for broadcasting: (N, 1, 4) vs (1, M, 4)
    a = boxes_a[:, np.newaxis, :]  # (N, 1, 4)
    b = boxes_b[np.newaxis, :, :]  # (1, M, 4)
    
    # Intersection
    inter_x1 = np.maximum(a[..., 0], b[..., 0])  # (N, M)
    inter_y1 = np.maximum(a[..., 1], b[..., 1])
    inter_x2 = np.minimum(a[..., 2], b[..., 2])
    inter_y2 = np.minimum(a[..., 3], b[..., 3])
    
    inter_w = np.maximum(0, inter_x2 - inter_x1)
    inter_h = np.maximum(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h  # (N, M)
    
    # Areas
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])  # (N,)
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])  # (M,)
    
    # Union: (N, 1) + (1, M) - (N, M)
    union_area = area_a[:, np.newaxis] + area_b[np.newaxis, :] - inter_area
    
    # IoU
    iou_matrix = np.where(union_area > 0, inter_area / union_area, 0.0)
    return iou_matrix


# Test
gt_boxes = np.array([
    [30, 50, 170, 200],   # cat
    [200, 80, 350, 250],  # dog
    [280, 20, 370, 90],   # bird
])

pred_boxes = np.array([
    [25, 45, 175, 205],   # pred cat
    [195, 75, 355, 255],  # pred dog
    [275, 15, 375, 95],   # pred bird
    [100, 150, 200, 270], # false positive
])

iou_mat = compute_iou_matrix(pred_boxes, gt_boxes)

print("IoU Matrix (predictions x ground_truth):")
print(f"{'':>15s}  {'cat':>8s}  {'dog':>8s}  {'bird':>8s}")
pred_labels = ['pred cat', 'pred dog', 'pred bird', 'false pos']
for i, label in enumerate(pred_labels):
    row = '  '.join(f"{iou_mat[i, j]:.4f}" for j in range(3))
    print(f"{label:>15s}  {row}")

### Solution 3: Simple NMS Implementation

In [ ]:
def non_maximum_suppression(detections, iou_threshold=0.5):
    """Apply Non-Maximum Suppression to filter overlapping detections.
    
    Args:
        detections: list of dicts with 'bbox' [x1,y1,x2,y2] and 'confidence'
        iou_threshold: IoU threshold above which to suppress
    
    Returns:
        kept: list of detections that survived NMS
    """
    if not detections:
        return []
    
    # Sort by confidence (highest first)
    sorted_dets = sorted(detections, key=lambda d: d['confidence'], reverse=True)
    
    kept = []
    
    while sorted_dets:
        # Keep the highest-confidence detection
        best = sorted_dets.pop(0)
        kept.append(best)
        
        # Remove detections that overlap too much with the kept one
        remaining = []
        for det in sorted_dets:
            iou = compute_iou(best['bbox'], det['bbox'])
            if iou < iou_threshold:
                remaining.append(det)  # Keep: not overlapping enough
            else:
                pass  # Suppress: too much overlap
        sorted_dets = remaining
    
    return kept


# Test with overlapping detections
test_detections = [
    {'label': 'cat', 'bbox': [30, 50, 170, 200],  'confidence': 0.92},
    {'label': 'cat', 'bbox': [35, 55, 175, 205],  'confidence': 0.85},  # Overlaps with above
    {'label': 'cat', 'bbox': [32, 48, 168, 198],  'confidence': 0.78},  # Overlaps with above
    {'label': 'dog', 'bbox': [200, 80, 350, 250], 'confidence': 0.88},
    {'label': 'dog', 'bbox': [205, 85, 355, 255], 'confidence': 0.70},  # Overlaps with above
]

print(f"Before NMS: {len(test_detections)} detections")
for d in test_detections:
    print(f"  {d['label']} conf={d['confidence']:.2f} bbox={d['bbox']}")

kept = non_maximum_suppression(test_detections, iou_threshold=0.5)

print(f"\nAfter NMS:  {len(kept)} detections")
for d in kept:
    print(f"  {d['label']} conf={d['confidence']:.2f} bbox={d['bbox']}")

---

**Next notebook**: [07 - Explainability with Grad-CAM (Optional)](07_Explainability_GradCAM_optional.ipynb)